# WoWAH Forecasting and Trend Detection

This notebook builds the forecasting and trend monitoring layer for PlayerPulse LiveOps from the completed aggregate marts. It uses the full-data outputs only and does not read preview files or rebuild raw staging.

Inputs:
- `data/processed/agg_daily_metrics.parquet`
- `data/processed/agg_cohort_retention.parquet`
- `data/processed/agg_segment_monthly.parquet`

Outputs:
- `data/processed/mart_forecast_daily.parquet`
- `data/processed/mart_forecast_backtest.parquet`
- `data/processed/mart_trend_alerts.parquet`
- `data/outputs/forecasting_validation_summary.csv`

## Method

Simple baseline models are used before complex models because they are transparent, fast to validate, and provide a defensible benchmark for operational LiveOps monitoring. If a future model cannot beat these baselines in rolling backtests, it should not be promoted.

Rolling-origin backtesting is used because this is time-series data. Random train/test splits would leak future seasonality and produce misleading accuracy estimates.

Limitations:
- No marketing campaign calendar is available.
- No game patch, content release, or event metadata is available.
- No revenue or monetization data is available.
- Forecasts target player activity, not financial performance.

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 80)

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

processed_dir = project_root / 'data' / 'processed'
outputs_dir = project_root / 'data' / 'outputs'
sql_path = project_root / 'sql' / '05_forecasting_trend_detection.sql'

agg_daily_metrics_path = processed_dir / 'agg_daily_metrics.parquet'
agg_cohort_retention_path = processed_dir / 'agg_cohort_retention.parquet'
agg_segment_monthly_path = processed_dir / 'agg_segment_monthly.parquet'

forecast_daily_path = processed_dir / 'mart_forecast_daily.parquet'
forecast_backtest_path = processed_dir / 'mart_forecast_backtest.parquet'
trend_alerts_path = processed_dir / 'mart_trend_alerts.parquet'
validation_summary_path = outputs_dir / 'forecasting_validation_summary.csv'

required_inputs = [agg_daily_metrics_path, agg_cohort_retention_path, agg_segment_monthly_path, sql_path]
missing_inputs = [str(path.relative_to(project_root)) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f'Missing required inputs: {missing_inputs}')

print('Project root:', project_root)
print('Input source used:', agg_daily_metrics_path.relative_to(project_root))

## DuckDB Views

The SQL file creates a clean daily forecast input with a full calendar date spine, then builds trend alert views from daily metrics, cohort retention, and monthly segment movement.

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW agg_daily_metrics AS SELECT * FROM read_parquet('{agg_daily_metrics_path.as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW agg_cohort_retention AS SELECT * FROM read_parquet('{agg_cohort_retention_path.as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW agg_segment_monthly AS SELECT * FROM read_parquet('{agg_segment_monthly_path.as_posix()}')")
con.execute(sql_path.read_text())

source_validation = con.execute('''
    SELECT
        COUNT(*) AS daily_rows,
        MIN(activity_date) AS min_activity_date,
        MAX(activity_date) AS max_activity_date,
        COUNT(DISTINCT activity_date) AS distinct_activity_dates,
        SUM(CASE WHEN dau IS NULL THEN 1 ELSE 0 END) AS null_dau_rows,
        SUM(CASE WHEN dau < 0 THEN 1 ELSE 0 END) AS negative_dau_rows
    FROM forecast_input_daily
''').df()
source_validation

## Forecast Input

The model target is DAU. The input keeps WAU, MAU, stickiness, observations, lifecycle counts, and date features for validation and future model extensions.

In [ ]:
forecast_input = con.execute('SELECT * FROM forecast_input_daily ORDER BY activity_date').df()
forecast_input['activity_date'] = pd.to_datetime(forecast_input['activity_date'])
forecast_input['dau'] = forecast_input['dau'].astype(float)

if forecast_input['activity_date'].duplicated().any():
    raise ValueError('forecast_input_daily has duplicate activity_date rows')
if forecast_input['dau'].isna().any():
    raise ValueError('forecast_input_daily has null DAU values')
if (forecast_input['dau'] < 0).any():
    raise ValueError('forecast_input_daily has negative DAU values')

forecast_input.head()

## Baseline Models

The baselines forecast DAU using only historical values available at the end of each training window:
- `naive_1d`: latest observed DAU from the train window.
- `moving_avg_7d`: average DAU over the final 7 train days.
- `seasonal_naive_7d`: most recent same-weekday DAU available in the train window.
- `moving_avg_30d`: average DAU over the final 30 train days.

In [ ]:
MODEL_NAMES = ['naive_1d', 'moving_avg_7d', 'seasonal_naive_7d', 'moving_avg_30d']
MIN_TRAIN_DAYS = 180
FORECAST_HORIZON_DAYS = 30
BACKTEST_STEP_DAYS = 30


def _history_series(train_df: pd.DataFrame) -> pd.Series:
    series = train_df.set_index('activity_date')['dau'].sort_index().astype(float)
    if series.empty:
        raise ValueError('Training history is empty')
    return series


def forecast_from_train(train_df: pd.DataFrame, forecast_dates: list[pd.Timestamp], model_name: str) -> list[float]:
    series = _history_series(train_df)
    last_value = float(series.iloc[-1])

    if model_name == 'naive_1d':
        return [max(0.0, last_value) for _ in forecast_dates]

    if model_name == 'moving_avg_7d':
        value = float(series.tail(7).mean())
        return [max(0.0, value) for _ in forecast_dates]

    if model_name == 'moving_avg_30d':
        value = float(series.tail(30).mean())
        return [max(0.0, value) for _ in forecast_dates]

    if model_name == 'seasonal_naive_7d':
        forecasts = []
        for forecast_date in forecast_dates:
            candidate = forecast_date - pd.Timedelta(days=7)
            while candidate not in series.index and candidate >= series.index.min():
                candidate -= pd.Timedelta(days=7)
            if candidate in series.index:
                forecasts.append(max(0.0, float(series.loc[candidate])))
            else:
                forecasts.append(max(0.0, last_value))
        return forecasts

    raise ValueError(f'Unknown model: {model_name}')


def safe_mape(actual: pd.Series, forecast: pd.Series) -> float:
    actual = actual.astype(float)
    forecast = forecast.astype(float)
    mask = actual != 0
    if not mask.any():
        return float('nan')
    return float((np.abs((forecast[mask] - actual[mask]) / actual[mask])).mean())


def safe_smape(actual: pd.Series, forecast: pd.Series) -> float:
    actual = actual.astype(float)
    forecast = forecast.astype(float)
    denom = np.abs(actual) + np.abs(forecast)
    mask = denom != 0
    if not mask.any():
        return float('nan')
    return float((2.0 * np.abs(forecast[mask] - actual[mask]) / denom[mask]).mean())

## Rolling Backtest

Each backtest window trains through `train_end_date`, forecasts the next 30 days, and records actuals only for scoring. The forecasting functions do not consume actual values inside the test horizon.

In [ ]:
def build_backtest(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values('activity_date').reset_index(drop=True)
    rows: list[dict] = []
    max_idx = len(df) - 1
    train_end_idx = MIN_TRAIN_DAYS - 1

    while train_end_idx < max_idx:
        test_start_idx = train_end_idx + 1
        test_end_idx = min(test_start_idx + FORECAST_HORIZON_DAYS - 1, max_idx)
        train_df = df.iloc[: train_end_idx + 1].copy()
        test_df = df.iloc[test_start_idx : test_end_idx + 1].copy()
        forecast_dates = list(test_df['activity_date'])
        train_end_date = df.loc[train_end_idx, 'activity_date']
        test_start_date = test_df['activity_date'].min()
        test_end_date = test_df['activity_date'].max()

        for model_name in MODEL_NAMES:
            forecasts = forecast_from_train(train_df, forecast_dates, model_name)
            for forecast_date, actual_dau, forecast_dau in zip(forecast_dates, test_df['dau'], forecasts):
                error = float(forecast_dau) - float(actual_dau)
                pct_error = np.nan if actual_dau == 0 else error / float(actual_dau)
                rows.append({
                    'train_end_date': train_end_date.date(),
                    'test_start_date': test_start_date.date(),
                    'test_end_date': test_end_date.date(),
                    'model_name': model_name,
                    'forecast_date': forecast_date.date(),
                    'actual_dau': float(actual_dau),
                    'forecast_dau': float(forecast_dau),
                    'error': error,
                    'abs_error': abs(error),
                    'pct_error': pct_error,
                    'squared_error': error ** 2,
                })

        train_end_idx += BACKTEST_STEP_DAYS

    return pd.DataFrame(rows)


backtest_df = build_backtest(forecast_input)
if backtest_df.empty:
    raise ValueError('No backtest windows generated; input date range is too short')
if backtest_df['actual_dau'].isna().any():
    raise ValueError('Backtest contains null actual_dau values')
if (pd.to_datetime(backtest_df['forecast_date']) <= pd.to_datetime(backtest_df['train_end_date'])).any():
    raise ValueError('Backtest leakage check failed: forecast_date is not after train_end_date')

backtest_windows = backtest_df[['train_end_date', 'test_start_date', 'test_end_date']].drop_duplicates().shape[0]
backtest_windows, backtest_df.head()

In [ ]:
metrics_rows = []
for model_name, model_df in backtest_df.groupby('model_name'):
    actual = model_df['actual_dau'].astype(float)
    forecast = model_df['forecast_dau'].astype(float)
    metrics_rows.append({
        'model_name': model_name,
        'mae': float(model_df['abs_error'].mean()),
        'rmse': float(math.sqrt(model_df['squared_error'].mean())),
        'mape': safe_mape(actual, forecast),
        'smape': safe_smape(actual, forecast),
        'bias': float(model_df['error'].mean()),
        'backtest_rows': int(len(model_df)),
    })

model_metrics = pd.DataFrame(metrics_rows).sort_values(['smape', 'mae'], na_position='last').reset_index(drop=True)
model_metrics['model_rank'] = np.arange(1, len(model_metrics) + 1)
champion_model = str(model_metrics.loc[0, 'model_name'])
champion_metrics = model_metrics.loc[0].to_dict()
model_metrics

## Final 30-Day Forecast

The champion model is selected by lowest sMAPE, then lowest MAE. Simple uncertainty bounds use the champion model's recent backtest residual standard deviation.

In [ ]:
latest_input_date = forecast_input['activity_date'].max()
final_forecast_dates = [latest_input_date + pd.Timedelta(days=i) for i in range(1, FORECAST_HORIZON_DAYS + 1)]
final_values = forecast_from_train(forecast_input, final_forecast_dates, champion_model)

champion_residuals = backtest_df.loc[backtest_df['model_name'] == champion_model, 'error'].astype(float)
residual_std = float(champion_residuals.tail(180).std(ddof=1)) if len(champion_residuals) > 1 else 0.0
if math.isnan(residual_std):
    residual_std = 0.0

forecast_created_at = pd.Timestamp.utcnow().tz_localize(None)
champion_rank = int(model_metrics.loc[model_metrics['model_name'] == champion_model, 'model_rank'].iloc[0])
forecast_daily_df = pd.DataFrame({
    'forecast_created_at': forecast_created_at,
    'forecast_date': [d.date() for d in final_forecast_dates],
    'model_name': champion_model,
    'forecast_target': 'DAU',
    'forecast_value': [max(0.0, float(v)) for v in final_values],
    'model_rank': champion_rank,
    'is_champion_model': True,
})
forecast_daily_df['lower_bound_simple'] = np.maximum(0.0, forecast_daily_df['forecast_value'] - 1.96 * residual_std)
forecast_daily_df['upper_bound_simple'] = forecast_daily_df['forecast_value'] + 1.96 * residual_std
forecast_daily_df = forecast_daily_df[[
    'forecast_created_at', 'forecast_date', 'model_name', 'forecast_target', 'forecast_value',
    'lower_bound_simple', 'upper_bound_simple', 'model_rank', 'is_champion_model'
]]

if (pd.to_datetime(forecast_daily_df['forecast_date']) <= latest_input_date).any():
    raise ValueError('Final forecast dates must be after max input activity_date')
if (forecast_daily_df[['forecast_value', 'lower_bound_simple']] < 0).any().any():
    raise ValueError('Final forecast contains negative forecast or lower bound')

forecast_daily_df.head()

## Trend Alerts

Trend alerts combine daily activity movement, D7 retention movement, and monthly lifecycle segment movement. Only triggered alerts are written.

In [ ]:
trend_alerts_df = con.execute('SELECT * FROM mart_trend_alerts ORDER BY alert_date DESC, alert_type').df()
if not trend_alerts_df.empty:
    if trend_alerts_df[['current_value', 'baseline_value']].isna().any().any():
        raise ValueError('Trend alerts contain null current_value or baseline_value')

trend_alerts_df

## Persist Outputs

DuckDB writes parquet outputs so the notebook does not depend on pandas parquet engines.

In [ ]:
con.register('backtest_df', backtest_df)
con.register('forecast_daily_df', forecast_daily_df)
con.register('trend_alerts_df', trend_alerts_df)

con.execute(f"COPY (SELECT * FROM backtest_df) TO '{forecast_backtest_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY (SELECT * FROM forecast_daily_df) TO '{forecast_daily_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY (SELECT * FROM trend_alerts_df) TO '{trend_alerts_path.as_posix()}' (FORMAT PARQUET)")

print('Wrote:', forecast_backtest_path.relative_to(project_root))
print('Wrote:', forecast_daily_path.relative_to(project_root))
print('Wrote:', trend_alerts_path.relative_to(project_root))

## Validation Summary

The summary CSV stores source range, backtest/model quality, final forecast coverage, and triggered alert counts.

In [ ]:
input_min_date = forecast_input['activity_date'].min().date()
input_max_date = forecast_input['activity_date'].max().date()
forecast_start_date = forecast_daily_df['forecast_date'].min()
forecast_end_date = forecast_daily_df['forecast_date'].max()

warnings_list: list[str] = []
missing_calendar_days = pd.date_range(forecast_input['activity_date'].min(), forecast_input['activity_date'].max(), freq='D').difference(forecast_input['activity_date'])
if len(missing_calendar_days) > 0:
    warnings_list.append(f'missing_calendar_days_after_spine:{len(missing_calendar_days)}')
if backtest_df['pct_error'].isna().any():
    warnings_list.append('mape_excludes_zero_actual_dau_rows')
if trend_alerts_df.empty:
    warnings_list.append('no_trend_alerts_triggered')

alert_type_counts = trend_alerts_df['alert_type'].value_counts().to_dict() if not trend_alerts_df.empty else {}
alert_severity_counts = trend_alerts_df['severity'].value_counts().to_dict() if not trend_alerts_df.empty else {}

summary_rows = [
    {'section': 'input', 'metric_name': 'input_source', 'metric_value': str(agg_daily_metrics_path.relative_to(project_root))},
    {'section': 'input', 'metric_name': 'input_date_range', 'metric_value': f'{input_min_date} to {input_max_date}'},
    {'section': 'input', 'metric_name': 'number_of_daily_rows', 'metric_value': int(len(forecast_input))},
    {'section': 'backtest', 'metric_name': 'number_of_backtest_windows', 'metric_value': int(backtest_windows)},
    {'section': 'backtest', 'metric_name': 'models_evaluated', 'metric_value': ', '.join(MODEL_NAMES)},
    {'section': 'champion', 'metric_name': 'champion_model', 'metric_value': champion_model},
    {'section': 'champion', 'metric_name': 'champion_mae', 'metric_value': champion_metrics['mae']},
    {'section': 'champion', 'metric_name': 'champion_rmse', 'metric_value': champion_metrics['rmse']},
    {'section': 'champion', 'metric_name': 'champion_mape', 'metric_value': champion_metrics['mape']},
    {'section': 'champion', 'metric_name': 'champion_smape', 'metric_value': champion_metrics['smape']},
    {'section': 'champion', 'metric_name': 'champion_bias', 'metric_value': champion_metrics['bias']},
    {'section': 'forecast', 'metric_name': 'final_forecast_start_date', 'metric_value': str(forecast_start_date)},
    {'section': 'forecast', 'metric_name': 'final_forecast_end_date', 'metric_value': str(forecast_end_date)},
    {'section': 'forecast', 'metric_name': 'number_of_forecast_rows', 'metric_value': int(len(forecast_daily_df))},
    {'section': 'alerts', 'metric_name': 'number_of_trend_alerts', 'metric_value': int(len(trend_alerts_df))},
    {'section': 'warnings', 'metric_name': 'warnings', 'metric_value': '; '.join(warnings_list) if warnings_list else 'none'},
]

for alert_type, count in alert_type_counts.items():
    summary_rows.append({'section': 'alerts_by_type', 'metric_name': alert_type, 'metric_value': int(count)})
for severity, count in alert_severity_counts.items():
    summary_rows.append({'section': 'alerts_by_severity', 'metric_name': severity, 'metric_value': int(count)})

for _, row in model_metrics.iterrows():
    summary_rows.append({'section': 'model_metrics', 'metric_name': f"{row['model_name']}_mae", 'metric_value': row['mae']})
    summary_rows.append({'section': 'model_metrics', 'metric_name': f"{row['model_name']}_rmse", 'metric_value': row['rmse']})
    summary_rows.append({'section': 'model_metrics', 'metric_name': f"{row['model_name']}_mape", 'metric_value': row['mape']})
    summary_rows.append({'section': 'model_metrics', 'metric_name': f"{row['model_name']}_smape", 'metric_value': row['smape']})
    summary_rows.append({'section': 'model_metrics', 'metric_name': f"{row['model_name']}_bias", 'metric_value': row['bias']})

validation_summary_df = pd.DataFrame(summary_rows)
validation_summary_df.to_csv(validation_summary_path, index=False)
validation_summary_df

## Final Report Values

This cell prints the exact fields needed for handoff reporting.

In [ ]:
top_10_recent_alerts = trend_alerts_df.sort_values(['alert_date', 'alert_type'], ascending=[False, True]).head(10)
final_report = {
    'Input source used': str(agg_daily_metrics_path.relative_to(project_root)),
    'Date range': f'{input_min_date} to {input_max_date}',
    'Number of daily rows': int(len(forecast_input)),
    'Models evaluated': ', '.join(MODEL_NAMES),
    'Backtest windows': int(backtest_windows),
    'Champion model': champion_model,
    'Champion MAE': float(champion_metrics['mae']),
    'Champion RMSE': float(champion_metrics['rmse']),
    'Champion MAPE': float(champion_metrics['mape']) if not pd.isna(champion_metrics['mape']) else None,
    'Champion sMAPE': float(champion_metrics['smape']) if not pd.isna(champion_metrics['smape']) else None,
    'Final forecast date range': f'{forecast_start_date} to {forecast_end_date}',
    'Number of trend alerts': int(len(trend_alerts_df)),
    'Files created': [
        str(forecast_daily_path.relative_to(project_root)),
        str(forecast_backtest_path.relative_to(project_root)),
        str(trend_alerts_path.relative_to(project_root)),
        str(validation_summary_path.relative_to(project_root)),
    ],
    'Any warnings': warnings_list or ['none'],
}

print(final_report)
print('\nTop 10 most recent alerts:')
display(top_10_recent_alerts)